In [5]:
!unzip -oq data.zip
!pip install -q gradio

import gradio as gr
import pandas as pd
import numpy as np
import joblib

# 1. Nạp mô hình tối ưu và tập kiểm thử độc lập (N = 60)
model = joblib.load("best_lr_pipeline.joblib")
X_test = pd.read_csv("data/X_test_raw.csv")
y_test = np.load("data/y_test.npy")

patient_choices = [
    f"Bệnh nhân #{i+1:02d} — Thực tế: {'Có bệnh (1)' if y_test[i]==1 else 'Không bệnh (0)'}"
    for i in range(len(X_test))
]

# Hàm dự đoán từ 13 chỉ số
def predict_heart_disease(age, sex, cp, trestbps, chol, fbs, restecg,
                          thalach, exang, oldpeak, slope, ca, thal, threshold):
    input_df = pd.DataFrame([{
        "age": age, "sex": sex, "cp": cp, "trestbps": trestbps, "chol": chol,
        "fbs": fbs, "restecg": restecg, "thalach": thalach, "exang": exang,
        "oldpeak": oldpeak, "slope": slope, "ca": ca, "thal": thal
    }])

    prob = float(model.predict_proba(input_df)[0, 1])
    pred = 1 if prob >= threshold else 0

    prob_text = f"{prob * 100:.2f}%"
    if pred == 1:
        status = f"⚠️ NGUY CƠ MẮC BỆNH TIM (Lớp 1 — Vượt ngưỡng {threshold:.2f})"
        advice = (
            f"Xác suất dự đoán đạt {prob*100:.2f}% (≥ ngưỡng {threshold:.2f}). "
            "Khuyến nghị chuyển tuyến chuyên khoa tim mạch để thăm khám và làm xét nghiệm chuyên sâu."
        )
    else:
        status = f"✅ KHÔNG MẮC BỆNH TIM (Lớp 0 — Dưới ngưỡng {threshold:.2f})"
        advice = (
            f"Xác suất dự đoán ở mức {prob*100:.2f}% (< ngưỡng {threshold:.2f}). "
            "Nguy cơ thấp theo mô hình sàng lọc; khuyến nghị duy trì lối sống lành mạnh và theo dõi định kỳ."
        )
    return prob_text, status, advice

# Hàm tự động nạp dữ liệu VÀ dự đoán luôn khi đổi lựa chọn trên Dropdown
def on_select_patient(choice_str, threshold):
    idx = patient_choices.index(choice_str)
    row = X_test.loc[idx]
    true_lbl = "⚠️ Có bệnh tim (1)" if y_test[idx] == 1 else "✅ Không mắc bệnh (0)"

    prob_text, status, advice = predict_heart_disease(
        float(row["age"]), float(row["sex"]), float(row["cp"]),
        float(row["trestbps"]), float(row["chol"]), float(row["fbs"]),
        float(row["restecg"]), float(row["thalach"]), float(row["exang"]),
        float(row["oldpeak"]), float(row["slope"]), float(row["ca"]),
        float(row["thal"]), threshold
    )
    return (
        float(row["age"]), float(row["sex"]), float(row["cp"]),
        float(row["trestbps"]), float(row["chol"]), float(row["fbs"]),
        float(row["restecg"]), float(row["thalach"]), float(row["exang"]),
        float(row["oldpeak"]), float(row["slope"]), float(row["ca"]),
        float(row["thal"]), true_lbl, prob_text, status, advice
    )

# 2. Thiết kế giao diện Web Demo
with gr.Blocks(theme=gr.themes.Soft(), title="Demo Dự đoán Nguy cơ Bệnh tim") as demo:
    gr.Markdown(
        """
        # ❤️ HỆ THỐNG HỖ TRỢ SÀNG LỌC NGUY CƠ BỆNH TIM
        **Mô hình lõi:** Logistic Regression (`C = 0.316`, điều chuẩn `L2`) | **Hiệu năng 5-Fold CV:** `ROC-AUC = 0.903`, `Accuracy = 83.5%`
        """
    )

    with gr.Row():
        patient_dropdown = gr.Dropdown(
            choices=patient_choices, value=patient_choices[0],
            label="Chọn hồ sơ mẫu từ tập Test độc lập (N = 60) — Tự động nạp & dự đoán", scale=3
        )
        true_label_box = gr.Textbox(
            value="✅ Không mắc bệnh (0)",
            label="Chẩn đoán thực tế (Bệnh viện)", interactive=False, scale=2
        )

    gr.Markdown("### 1. Nhập hoặc điều chỉnh 13 chỉ số lâm sàng")
    with gr.Row():
        with gr.Column():
            age = gr.Number(value=float(X_test.loc[0, "age"]), label="Tuổi (age)")
            sex = gr.Dropdown(choices=[("1 - Nam", 1.0), ("0 - Nữ", 0.0)], value=float(X_test.loc[0, "sex"]), label="Giới tính (sex)")
            cp = gr.Dropdown(
                choices=[("1 - Đau thắt ngực điển hình", 1.0), ("2 - Đau không điển hình", 2.0),
                         ("3 - Đau không do tim", 3.0), ("4 - Không triệu chứng", 4.0)],
                value=float(X_test.loc[0, "cp"]), label="Loại đau ngực (cp)"
            )
            trestbps = gr.Number(value=float(X_test.loc[0, "trestbps"]), label="Huyết áp khi nghỉ (trestbps, mmHg)")
            chol = gr.Number(value=float(X_test.loc[0, "chol"]), label="Cholesterol huyết thanh (chol, mg/dl)")

        with gr.Column():
            fbs = gr.Dropdown(choices=[("0 - Không (≤ 120 mg/dl)", 0.0), ("1 - Có (> 120 mg/dl)", 1.0)], value=float(X_test.loc[0, "fbs"]), label="Đường huyết lúc đói (fbs)")
            restecg = gr.Dropdown(
                choices=[("0 - Bình thường", 0.0), ("1 - Bất thường sóng ST-T", 1.0), ("2 - Phì đại thất trái", 2.0)],
                value=float(X_test.loc[0, "restecg"]), label="Điện tâm đồ khi nghỉ (restecg)"
            )
            thalach = gr.Number(value=float(X_test.loc[0, "thalach"]), label="Nhịp tim tối đa (thalach)")
            exang = gr.Dropdown(choices=[("0 - Không", 0.0), ("1 - Có", 1.0)], value=float(X_test.loc[0, "exang"]), label="Đau thắt ngực khi gắng sức (exang)")

        with gr.Column():
            oldpeak = gr.Number(value=float(X_test.loc[0, "oldpeak"]), step=0.1, label="Độ chênh xuống đoạn ST (oldpeak)")
            slope = gr.Dropdown(
                choices=[("1 - Dốc lên", 1.0), ("2 - Đi ngang", 2.0), ("3 - Dốc xuống", 3.0)],
                value=float(X_test.loc[0, "slope"]), label="Độ dốc đoạn ST (slope)"
            )
            ca = gr.Dropdown(choices=[0.0, 1.0, 2.0, 3.0], value=float(X_test.loc[0, "ca"]), label="Số mạch máu lớn quan sát được (ca)")
            thal = gr.Dropdown(
                choices=[("3 - Bình thường", 3.0), ("6 - Khuyết tật cố định", 6.0), ("7 - Khuyết tật hồi phục được", 7.0)],
                value=float(X_test.loc[0, "thal"]), label="Kết quả xét nghiệm Thal (thal)"
            )
            threshold = gr.Slider(0.10, 0.90, value=0.50, step=0.05, label="Ngưỡng quyết định phân loại (Threshold)")

    predict_btn = gr.Button("🔍 PHÂN TÍCH & DỰ ĐOÁN LẠI (SAU KHI SỬA CHỈ SỐ)", variant="primary")

    gr.Markdown("### 2. Kết quả chẩn đoán từ mô hình")
    with gr.Row():
        out_prob = gr.Textbox(label="Xác suất mắc bệnh tim P(y = 1 | x)", interactive=False)
        out_status = gr.Textbox(label="Kết luận phân loại", interactive=False)
    out_advice = gr.Textbox(label="Khuyến nghị sàng lọc lâm sàng", lines=2, interactive=False)

    inputs_list = [age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal]

    # Tự động cập nhật ngay khi chọn bệnh nhân khác trên Dropdown
    patient_dropdown.change(
        fn=on_select_patient,
        inputs=[patient_dropdown, threshold],
        outputs=inputs_list + [true_label_box, out_prob, out_status, out_advice]
    )
    # Tự động tính lại khi kéo thanh trượt Threshold
    threshold.change(
        fn=predict_heart_disease,
        inputs=inputs_list + [threshold],
        outputs=[out_prob, out_status, out_advice]
    )
    # Bấm nút khi tự nhập/sửa tay các chỉ số
    predict_btn.click(
        fn=predict_heart_disease,
        inputs=inputs_list + [threshold],
        outputs=[out_prob, out_status, out_advice]
    )

demo.launch(share=True)

/tmp/ipykernel_764/496952827.py:68: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Demo Dự đoán Nguy cơ Bệnh tim") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59dffc37fbcabe2e1c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
